# P5 — Evaluation notebook (uses your already-trained models)
No training here. This loads `yolo_car_best.pt`, `classifier_best.pth` and `classical_lr.joblib` and completes the remaining requirements:

| PS §4 requirement | Section |
|---|---|
| 1. Distinguish visually similar makes / models / years | **A** — error hierarchy, most-confused pairs, year confusion |
| 2. Varying illumination & viewing angles | **B** — illumination perturbations + viewpoint slices |
| 3. Partial occlusion & different resolutions | **B** — occlusion 25/40 %, ×0.25 / ×0.1 resolution, blur, noise, rotation |
| 6. Model optimisation | **D** — FP16, ONNX, INT8 quantisation, pruning |
| 8. ≥ 3 failure cases with causes | **E** — cause tagging, Grad-CAM gallery |
| Confidence threshold analysis (§5) | **C** — calibration, ECE, fixed thresholds |
| Frontend files | **F** — `config.json`, `inference.py`, end-to-end test |
| 4, 5, 7 | already in your `metrics.json` (kept and merged) |

**Setup:** `Runtime → Change runtime type → T4 GPU`, then **Run all** (~60 min). Results are merged into `artifacts/metrics.json` on Drive; intermediate predictions are cached on Drive so a disconnect doesn't restart everything.

Your trained files should be in `MyDrive/P5_vehicle_recognition/artifacts/`. If they aren't, cell 1 asks you to upload `P5_artifacts.zip`.

In [ ]:
# @title 0. Installs
!nvidia-smi -L || echo "NO GPU — switch runtime to T4 GPU"
!pip -q install ultralytics timm onnx onnxruntime onnxscript scikit-image

In [ ]:
# @title 1. Config, Drive, locate trained files
import os, sys, json, time, math, random, glob, shutil, copy, warnings, subprocess, zipfile
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm, joblib
from scipy.special import log_softmax, softmax
from multiprocessing import Pool
cv2.setNumThreads(0)

CFG = dict(
    SEED=42, VAL_FRAC=0.10, CROP_PAD=0.08,
    DET_CONF=0.25, YOLO_IMGSZ=640,
    N_ROBUST=1000,          # test images per robustness condition
    N_OPT_EVAL=1000,        # test images for optimisation accuracy
    TARGET_PRECISION=0.99,  # accuracy required for "confident" predictions
    TAU_MAKE_TARGET=0.98,   # make-level accuracy required for "uncertain_model"
    MIN_TAU=0.30,           # thresholds never go below this
    DARK_THR=80, BRIGHT_THR=200, NUM_WORKERS=2,
)
SEED, NW = CFG['SEED'], CFG['NUM_WORKERS']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/P5_vehicle_recognition'
except Exception:
    OUT = '/content/P5_vehicle_recognition'
ART, PLOTS, CACHE, WORK = f'{OUT}/artifacts', f'{OUT}/plots', f'{OUT}/eval_cache', '/content/work'
for d in [ART, PLOTS, CACHE, WORK]: os.makedirs(d, exist_ok=True)
sys.path.insert(0, WORK)

NEEDED = ['yolo_car_best.pt', 'classifier_best.pth', 'classical_lr.joblib']
if not all(os.path.exists(f'{ART}/{f}') for f in NEEDED):
    print('Trained files not found in', ART, '-> upload P5_artifacts.zip')
    from google.colab import files
    for fn in files.upload():
        with zipfile.ZipFile(fn) as z: z.extractall(ART)
    # zip may contain a top-level folder
    for f in NEEDED:
        if not os.path.exists(f'{ART}/{f}'):
            hit = glob.glob(f'{ART}/**/{f}', recursive=True)
            if hit: shutil.copy(hit[0], ART)
for f in NEEDED:
    assert os.path.exists(f'{ART}/{f}'), f'Missing {f}'
    print(f'{f:24s} {os.path.getsize(f"{ART}/{f}")/1e6:7.1f} MB')

RESULTS_PATH = f'{ART}/metrics.json'
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
print('Existing metrics keys:', list(RESULTS))
def _js(o):
    if hasattr(o, 'item'): return o.item()
    if hasattr(o, 'tolist'): return o.tolist()
    return str(o)
def save_results(): json.dump(RESULTS, open(RESULTS_PATH, 'w'), indent=2, default=_js)
def savefig(name): plt.savefig(f'{PLOTS}/{name}.png', dpi=130, bbox_inches='tight')

In [ ]:
# @title 2. Dataset (same split as training: seed 42, 10 % val)
import scipy.io as sio
from sklearn.model_selection import train_test_split
DATA_REPO = '/content/StanfordCars-Dataset'
if not os.path.exists(DATA_REPO):
    !git clone --depth 1 https://github.com/jhpohovey/StanfordCars-Dataset.git {DATA_REPO}
def find_file(name): return glob.glob(f'{DATA_REPO}/**/{name}', recursive=True)[0]
def find_img_dir(name):
    for d in glob.glob(f'{DATA_REPO}/**/{name}', recursive=True):
        if os.path.isdir(d) and glob.glob(d + '/*.jpg'): return d
TRAIN_IMG_DIR, TEST_IMG_DIR = find_img_dir('cars_train'), find_img_dir('cars_test')
_probe = sorted(glob.glob(TRAIN_IMG_DIR + '/*.jpg'))[0]
if cv2.imread(_probe) is None:
    subprocess.run(f'cd {DATA_REPO} && git lfs install && git lfs pull', shell=True)

MULTI_WORD_MAKES = ['AM General', 'Aston Martin', 'Land Rover']
def parse_name(n):
    parts = n.split(); year = parts[-1] if parts[-1].isdigit() else ''
    body = ' '.join(parts[:-1] if year else parts)
    for mk in MULTI_WORD_MAKES:
        if body.startswith(mk + ' '): return mk, body[len(mk) + 1:], year
    mk, _, rest = body.partition(' '); return mk, rest, year
CLASS_NAMES = [str(x) for x in sio.loadmat(find_file('cars_meta.mat'), squeeze_me=True)['class_names']]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_META = [dict(id=i, name=n, **dict(zip(['make', 'model', 'year'], parse_name(n)))) for i, n in enumerate(CLASS_NAMES)]
MAKES = sorted({c['make'] for c in CLASS_META})
MAKE_ID = np.array([MAKES.index(c['make']) for c in CLASS_META])
MM_NAMES = sorted({c['make'] + ' ' + c['model'] for c in CLASS_META})
MM_ID = np.array([MM_NAMES.index(c['make'] + ' ' + c['model']) for c in CLASS_META])

def load_annos(path, img_dir):
    a = sio.loadmat(path, squeeze_me=True)['annotations']
    df = pd.DataFrame({'fname': [os.path.basename(str(f)) for f in a['fname']],
        'x1': a['bbox_x1'].astype(float), 'y1': a['bbox_y1'].astype(float),
        'x2': a['bbox_x2'].astype(float), 'y2': a['bbox_y2'].astype(float), 'label': a['class'].astype(int) - 1})
    df['path'] = img_dir + '/' + df.fname
    sizes = [Image.open(p).size for p in tqdm(df.path, desc='sizes', leave=False)]
    df['W'] = [s[0] for s in sizes]; df['H'] = [s[1] for s in sizes]
    return df
full_train_df = load_annos(find_file('cars_train_annos.mat'), TRAIN_IMG_DIR)
test_df = load_annos(find_file('cars_test_annos_withlabels.mat'), TEST_IMG_DIR)
train_df, val_df = train_test_split(full_train_df, test_size=CFG['VAL_FRAC'], stratify=full_train_df.label, random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
ytr, yval, yte = train_df.label.values, val_df.label.values, test_df.label.values
BOX = ['x1', 'y1', 'x2', 'y2']
GT_TEST, GT_VAL = test_df[BOX].values, val_df[BOX].values
print('train', len(train_df), '| val', len(val_df), '| test', len(test_df))

def cls_metrics(scores, y):
    scores, y = np.asarray(scores), np.asarray(y)
    pred = scores.argmax(1); top5 = np.argsort(-scores, 1)[:, :5]
    return dict(top1=float((pred == y).mean()), top5=float((top5 == y[:, None]).any(1).mean()),
                make_acc=float((MAKE_ID[pred] == MAKE_ID[y]).mean()), make_model_acc=float((MM_ID[pred] == MM_ID[y]).mean()))

In [ ]:
%%writefile /content/work/classical_cv.py
"""Classical computer-vision module for P5 fine-grained vehicle recognition."""
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

HOG_SIZE = (160, 96)  # (W, H) — cars are wider than tall


# ---------------- preprocessing / normalisation ----------------
def to_bgr(img):
    """PIL image, RGB array (from PIL) or BGR array (from cv2) / file path -> BGR uint8."""
    if isinstance(img, str):
        im = cv2.imread(img)
        if im is None:
            raise ValueError(f'Cannot read {img}')
        return im
    if hasattr(img, 'mode'):
        img = np.array(img.convert('RGB'))[:, :, ::-1]
    img = np.ascontiguousarray(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img


def resize_max(bgr, max_side):
    h, w = bgr.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        bgr = cv2.resize(bgr, (max(1, int(round(w * s))), max(1, int(round(h * s)))), interpolation=cv2.INTER_AREA)
    return bgr


def apply_clahe(bgr, clip=2.0, grid=8):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


def brightness(bgr):
    return float(cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[:, :, 2].mean())


def adaptive_enhance(bgr, dark_thr=80, bright_thr=200):
    """Illumination normalisation: gamma-lift + CLAHE for dark images, CLAHE for washed-out ones."""
    v = brightness(bgr)
    if v < dark_thr:
        gamma = max(0.4, v / dark_thr)
        lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
        return apply_clahe(cv2.LUT(bgr, lut), clip=2.5), 'dark: gamma+CLAHE'
    if v > bright_thr:
        return apply_clahe(bgr, clip=1.5), 'bright: CLAHE'
    return bgr, 'none'


def normalize_image(bgr):
    """Per-channel z-score normalisation; returns float image and a displayable uint8 version."""
    f = bgr.astype(np.float32)
    mean, std = f.reshape(-1, 3).mean(0), f.reshape(-1, 3).std(0) + 1e-6
    z = (f - mean) / std
    disp = cv2.normalize(z, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return z, disp


# ---------------- edges / contours / ROI ----------------
def auto_canny(gray, sigma=0.33):
    v = float(np.median(gray))
    return cv2.Canny(gray, int(max(0, (1 - sigma) * v)), int(min(255, (1 + sigma) * v)))


def edge_contour_analysis(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = auto_canny(blur)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, k, iterations=2)
    closed = cv2.dilate(closed, k, iterations=1)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return dict(gray=gray, blur=blur, edges=edges, closed=closed, contours=cnts)


def classical_roi(img, max_side=512):
    """Classical vehicle localisation: CLAHE -> Canny -> closing -> best (large, central, car-shaped) contour.
    Returns (box_xyxy in original coords or None, score in [0,1], debug dict)."""
    bgr = to_bgr(img)
    H, W = bgr.shape[:2]
    small = apply_clahe(resize_max(bgr, max_side))
    s = small.shape[1] / W
    ea = edge_contour_analysis(small)
    h, w = small.shape[:2]
    best, best_score = None, 0.0
    for c in ea['contours']:
        x, y, cw, ch = cv2.boundingRect(c)
        area = cw * ch / (w * h)
        if area < 0.05:
            continue
        cx, cy = (x + cw / 2) / w, (y + ch / 2) / h
        centrality = 1 - min(1.0, np.hypot(cx - 0.5, cy - 0.5) / 0.71)
        ar = cw / max(ch, 1)
        ar_score = 1.0 if 0.8 <= ar <= 3.5 else 0.5
        score = area * (0.5 + 0.5 * centrality) * ar_score
        if score > best_score:
            best_score, best = score, (x, y, x + cw, y + ch)
    if best is None:
        return None, 0.0, ea
    return tuple(v / s for v in best), float(min(1.0, best_score)), ea


def crop_box(bgr, box, pad=0.0):
    if box is None:
        return bgr
    H, W = bgr.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in box]
    bw, bh = x2 - x1, y2 - y1
    x1, y1 = int(max(0, x1 - pad * bw)), int(max(0, y1 - pad * bh))
    x2, y2 = int(min(W, x2 + pad * bw)), int(min(H, y2 + pad * bh))
    if x2 - x1 < 8 or y2 - y1 < 8:
        return bgr
    return bgr[y1:y2, x1:x2]


# ---------------- feature extraction ----------------
def color_features(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    feats = []
    for ch, bins, rng in [(0, 18, 180), (1, 8, 256), (2, 8, 256)]:
        h = cv2.calcHist([hsv], [ch], None, [bins], [0, rng]).ravel()
        feats.append(h / (h.sum() + 1e-6))
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    feats.append(np.concatenate([lab.mean(0), lab.std(0)]) / 255.0)
    return np.concatenate(feats)


def texture_features(gray):
    g = cv2.resize(gray, (128, 128))
    feats = []
    for P, R in [(8, 1), (16, 2)]:
        lbp = local_binary_pattern(g, P, R, method='uniform')
        h, _ = np.histogram(lbp, bins=P + 2, range=(0, P + 2))
        feats.append(h / (h.sum() + 1e-6))
    q = (g // 4).astype(np.uint8)
    glcm = graycomatrix(q, distances=[1, 3], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                        levels=64, symmetric=True, normed=True)
    for p in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feats.append(graycoprops(glcm, p).ravel())
    return np.concatenate(feats)


def hog_features(gray, visualize=False):
    g = cv2.resize(gray, HOG_SIZE)
    return hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
               block_norm='L2-Hys', visualize=visualize, feature_vector=True)


def geometric_features(bgr):
    h, w = bgr.shape[:2]
    ea = edge_contour_analysis(resize_max(bgr, 256))
    edges, cnts = ea['edges'], ea['contours']
    hh, ww = edges.shape
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        area = cv2.contourArea(c)
        x, y, cw, ch = cv2.boundingRect(c)
        hull_a = cv2.contourArea(cv2.convexHull(c)) + 1e-6
        per = cv2.arcLength(c, True) + 1e-6
        f = [area / (hh * ww), area / (cw * ch + 1e-6), area / hull_a, 4 * np.pi * area / per ** 2, cw / max(ch, 1)]
        hu = cv2.HuMoments(cv2.moments(c)).ravel()
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    else:
        f, hu = [0.0] * 5, np.zeros(7)
    gx = cv2.Sobel(ea['blur'], cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(ea['blur'], cv2.CV_32F, 0, 1)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    oh, _ = np.histogram(ang % 180, bins=8, range=(0, 180), weights=mag)
    oh = oh / (oh.sum() + 1e-6)
    col = edges.mean(0)
    sym = float(np.clip(1 - np.abs(col - col[::-1]).mean() / (2 * col.mean() + 1e-6), 0, 1))
    # layout: [aspect, edge_density, n_contours/50, area_ratio, extent, solidity, circularity, contour_ar, hu*7, orient*8, symmetry]
    return np.concatenate([[w / max(h, 1), edges.mean() / 255.0, len(cnts) / 50.0], f, hu, oh, [sym]])


def extract_classical_features(img):
    """Full hand-crafted descriptor (~1.75k dims): HOG + colour + texture + geometry."""
    bgr = resize_max(to_bgr(img), 320)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    f = np.concatenate([hog_features(gray), color_features(bgr), texture_features(gray), geometric_features(bgr)])
    return np.nan_to_num(f.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)


# multiprocessing helpers (top-level so they pickle)
def features_from_path(args):
    path, box, pad = args
    bgr = cv2.imread(path)
    return extract_classical_features(crop_box(bgr, box, pad))


def roi_from_path(path):
    box, score, _ = classical_roi(cv2.imread(path))
    return box, score


# ---------------- visualisation of intermediate stages ----------------
def _rgb(im):
    return cv2.cvtColor(im, cv2.COLOR_GRAY2RGB) if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)


def _to_u8(a):
    a = a.astype(np.float32)
    a = (a - a.min()) / (a.max() - a.min() + 1e-6)
    return (a * 255).astype(np.uint8)


def processing_stages(img, box=None, max_side=640):
    """Returns ({stage_name: RGB uint8 image}, {feature_name: value}) for the app's 'intermediate results' panel.
    `box` (optional) is the detector box in ORIGINAL image coordinates."""
    src = to_bgr(img)
    W0 = src.shape[1]
    bgr = resize_max(src, max_side)
    s = bgr.shape[1] / W0
    st = {'Original': bgr}
    enh, mode = adaptive_enhance(bgr)
    st[f'Illumination normalisation ({mode})'] = enh
    _, nd = normalize_image(enh)
    st['Z-score normalised'] = nd
    ea = edge_contour_analysis(enh)
    st['Grayscale + Gaussian blur'] = ea['blur']
    st['Canny edges'] = ea['edges']
    st['Morphological closing'] = ea['closed']
    cimg = bgr.copy()
    cv2.drawContours(cimg, ea['contours'], -1, (0, 255, 0), 1)
    roi, rscore, _ = classical_roi(bgr)
    if roi is not None:
        cv2.rectangle(cimg, (int(roi[0]), int(roi[1])), (int(roi[2]), int(roi[3])), (255, 0, 0), 2)
    dbox = [v * s for v in box] if box is not None else None
    if dbox is not None:
        cv2.rectangle(cimg, (int(dbox[0]), int(dbox[1])), (int(dbox[2]), int(dbox[3])), (0, 0, 255), 2)
    st['Contours + ROI (blue=classical, red=detector)'] = cimg
    use = dbox if dbox is not None else roi
    crop = crop_box(bgr, use, 0.05) if use is not None else bgr
    st['Region of interest'] = crop
    g = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), (320, 192))
    _, hv = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)
    st['HOG (shape / gradients)'] = _to_u8(hv)
    st['LBP texture'] = _to_u8(local_binary_pattern(g, 8, 1, 'uniform'))
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hue = np.dstack([hsv[:, :, 0], np.full_like(hsv[:, :, 0], 255), np.full_like(hsv[:, :, 0], 255)])
    st['Hue map (colour)'] = cv2.cvtColor(hue, cv2.COLOR_HSV2BGR)
    gf = geometric_features(crop)
    c = crop[crop.shape[0] // 4: 3 * crop.shape[0] // 4, crop.shape[1] // 4: 3 * crop.shape[1] // 4]
    info = dict(brightness=round(brightness(bgr), 1), illumination_fix=mode,
                roi_aspect_ratio=round(float(gf[0]), 3), edge_density=round(float(gf[1]), 4),
                num_contours=len(ea['contours']), contour_solidity=round(float(gf[5]), 3),
                lr_symmetry=round(float(gf[-1]), 3),
                mean_rgb=[int(v) for v in c.reshape(-1, 3).mean(0)[::-1]],
                classical_roi_score=round(rscore, 3))
    return {k: _rgb(v) for k, v in st.items()}, info

In [ ]:
# @title 3. Load trained models
import importlib, classical_cv
importlib.reload(classical_cv)
from classical_cv import *
shutil.copy(f'{WORK}/classical_cv.py', ART)
from ultralytics import YOLO

YOLO_BEST, BEST = f'{ART}/yolo_car_best.pt', f'{ART}/classifier_best.pth'
det_ft = YOLO(YOLO_BEST)
ck = torch.load(BEST, map_location='cpu', weights_only=False)
MODEL_NAME, S, MEAN, STD = ck['model_name'], ck['img_size'], list(ck['mean']), list(ck['std'])
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=ck['num_classes'])
model.load_state_dict(ck['state_dict']); model = model.to(DEVICE).eval().to(memory_format=torch.channels_last)
bundle = joblib.load(f'{ART}/classical_lr.joblib')
print(MODEL_NAME, 'img', S, '| classes', ck['num_classes'])

eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])
class CarDS(Dataset):
    def __init__(self, paths, labels, boxes=None, pad=CFG['CROP_PAD'], enhance=False):
        self.paths, self.labels, self.boxes, self.pad, self.enhance = list(paths), np.asarray(labels), boxes, pad, enhance
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        bgr = cv2.imread(self.paths[i])
        if self.boxes is not None: bgr = crop_box(bgr, self.boxes[i], self.pad)
        if self.enhance: bgr, _ = adaptive_enhance(bgr, CFG['DARK_THR'], CFG['BRIGHT_THR'])
        return eval_tf(Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))), int(self.labels[i])

@torch.no_grad()
def get_logits(ds, bs=64, desc='eval'):
    """Returns (plain logits, flip-TTA logits)."""
    outs, outs_t = [], []
    for x, _ in tqdm(DataLoader(ds, bs, shuffle=False, num_workers=NW, pin_memory=True), desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            z = model(x).float(); zt = (z + model(torch.flip(x, dims=[3])).float()) / 2
        outs.append(z.cpu()); outs_t.append(zt.cpu())
    return torch.cat(outs).numpy(), torch.cat(outs_t).numpy()

def pool_map(fn, items, desc):
    with Pool(os.cpu_count()) as p:
        return list(tqdm(p.imap(fn, items, chunksize=16), total=len(items), desc=desc))
def brightness_from_path(args):
    p, b = args
    return brightness(crop_box(cv2.imread(p), b, 0.0))
def classical_logp(X):
    return bundle['lr'].predict_log_proba(bundle['pca'].transform(bundle['scaler'].transform(X)))
def iou_one(boxes, g):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    ix1 = np.maximum(boxes[:, 0], g[0]); iy1 = np.maximum(boxes[:, 1], g[1])
    ix2 = np.minimum(boxes[:, 2], g[2]); iy2 = np.minimum(boxes[:, 3], g[3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    a = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]); b = (g[2] - g[0]) * (g[3] - g[1])
    return inter / (a + b - inter + 1e-9)
def bench(fn, n=30, warm=5):
    for _ in range(warm): fn()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n): fn()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000

In [ ]:
# @title 4. Recompute test/val predictions with the final hybrid pipeline (cached on Drive)
PRED_CACHE = f'{CACHE}/predictions.npz'
if os.path.exists(PRED_CACHE):
    c = np.load(PRED_CACHE, allow_pickle=True)
    TEST_BOXES_YOLO = [list(b) for b in c['boxes']]; DET_MS = float(c['det_ms'])
    z_te, zv, Xte_yolo, Xval, TEST_BRIGHT = c['z_te'], c['zv'], c['Xte'], c['Xval'], c['bright']
    print('Loaded cached predictions')
else:
    # 4.1 detection on unseen test images (classical ROI fallback if YOLO finds nothing)
    TEST_BOXES_YOLO = []; t0 = time.time()
    for i in tqdm(range(0, len(test_df), 32), desc='YOLO'):
        chunk = list(test_df.path[i:i + 32])
        res = det_ft.predict(chunk, conf=0.01, iou=0.6, imgsz=CFG['YOLO_IMGSZ'], half=DEVICE == 'cuda', verbose=False)
        for p, r in zip(chunk, res):
            b, s = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
            if len(b): TEST_BOXES_YOLO.append(b[np.argmax(s)].tolist())
            else:
                rb = classical_roi(cv2.imread(p))[0]
                TEST_BOXES_YOLO.append(list(rb) if rb is not None else [0, 0, *Image.open(p).size])
    DET_MS = (time.time() - t0) / len(test_df) * 1000
    # 4.2 deep logits: test = YOLO crops + illumination normalisation (final pipeline); val = GT crops
    _, z_te = get_logits(CarDS(test_df.path, yte, boxes=TEST_BOXES_YOLO, enhance=True), desc='test')
    _, zv = get_logits(CarDS(val_df.path, yval, boxes=GT_VAL.tolist()), desc='val')
    # 4.3 classical features (for hybrid fusion)
    Xte_yolo = np.stack(pool_map(features_from_path, [(p, b, CFG['CROP_PAD']) for p, b in zip(test_df.path, TEST_BOXES_YOLO)], 'classical test'))
    Xval = np.stack(pool_map(features_from_path, [(p, b, CFG['CROP_PAD']) for p, b in zip(val_df.path, GT_VAL.tolist())], 'classical val'))
    # 4.4 brightness of each test vehicle crop (for slices / failure causes)
    TEST_BRIGHT = np.array(pool_map(brightness_from_path, [(p, b) for p, b in zip(test_df.path, GT_TEST.tolist())], 'brightness'))
    np.savez(PRED_CACHE, boxes=np.array(TEST_BOXES_YOLO), det_ms=DET_MS, z_te=z_te, zv=zv, Xte=Xte_yolo, Xval=Xval, bright=TEST_BRIGHT)

IOU_FT = np.array([iou_one(b, g)[0] for b, g in zip(TEST_BOXES_YOLO, GT_TEST)])
lp_te, lp_val = classical_logp(Xte_yolo), classical_logp(Xval)

def fit_temperature(logits, y):
    lt, yt = torch.tensor(logits, dtype=torch.float32), torch.tensor(y)
    logT = torch.zeros(1, requires_grad=True); o = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
    def closure():
        o.zero_grad(); l = F.cross_entropy(lt / logT.exp(), yt); l.backward(); return l
    o.step(closure); return float(logT.exp())

if 'calibration_params' in RESULTS:      # keep the values from the training notebook
    T_DEEP, ALPHA, T2 = (RESULTS['calibration_params'][k] for k in ['T', 'alpha', 'T2'])
else:
    T_DEEP = fit_temperature(zv, yval)
    alphas = np.round(np.linspace(0, 1, 21), 2)
    ALPHA = float(alphas[int(np.argmax([cls_metrics(log_softmax(zv / T_DEEP, 1) + a * lp_val, yval)['top1'] for a in alphas]))])
    T2 = fit_temperature(log_softmax(zv / T_DEEP, 1) + ALPHA * lp_val, yval)
    RESULTS['calibration_params'] = dict(T=T_DEEP, alpha=ALPHA, T2=T2)
def hybrid_probs(z, lp): return softmax((log_softmax(z / T_DEEP, 1) + ALPHA * lp) / T2, 1)
P_TEST, P_VAL = hybrid_probs(z_te, lp_te), hybrid_probs(zv, lp_val)
print(f'T={T_DEEP:.3f} alpha={ALPHA} T2={T2:.3f} | detection {DET_MS:.1f} ms/img | mean IoU {IOU_FT.mean():.3f}')
print('Hybrid on unseen test (should match your earlier ~94.7 % top-1):', {k: round(v, 4) for k, v in cls_metrics(P_TEST, yte).items()})
RESULTS['final_system_test'] = cls_metrics(P_TEST, yte); save_results()

## A. Requirement 1 — Distinguishing visually similar makes, models and years
Every error is placed in exactly one bucket: **wrong year only** (right make + model), **same make, wrong model**, or **wrong make**. Then the most-confused class pairs and the hardest classes.

In [ ]:
# @title A.1 Error hierarchy + most-confused pairs
pred = P_TEST.argmax(1); conf = P_TEST.max(1); n = len(yte)
same_make = MAKE_ID[pred] == MAKE_ID[yte]; same_mm = MM_ID[pred] == MM_ID[yte]; ok = pred == yte
hier = {'correct': ok.mean(), 'wrong year only (same make+model)': (same_mm & ~ok).mean(),
        'same make, wrong model': (same_make & ~same_mm).mean(), 'wrong make': (~same_make).mean()}
RESULTS['similar_class_analysis'] = dict(error_hierarchy=hier)
plt.figure(figsize=(8, 2.8)); plt.barh(list(hier)[1:], [v * 100 for v in list(hier.values())[1:]])
plt.xlabel('% of test images'); plt.title(f'Where the {100*(1-ok.mean()):.1f}% errors come from'); savefig('error_hierarchy'); plt.show()
print({k: f'{v*100:.2f}%' for k, v in hier.items()})

# year-confusion: classes that exist in more than one year for the same make+model
multi_year = pd.Series([CLASS_META[c]['make'] + ' ' + CLASS_META[c]['model'] for c in range(NUM_CLASSES)]).value_counts()
multi_year = multi_year[multi_year > 1]
my_mask = np.isin(MM_ID[yte], [MM_NAMES.index(m) for m in multi_year.index])
print(f'\n{len(multi_year)} make+model groups appear in multiple years ({my_mask.sum()} test images).')
print(f'Year accuracy on those images (given make+model correct): {(ok[my_mask & same_mm]).mean():.2%}')
RESULTS['similar_class_analysis']['multi_year_groups'] = int(len(multi_year))
RESULTS['similar_class_analysis']['year_acc_given_make_model'] = float(ok[my_mask & same_mm].mean())

from collections import Counter
wrong = np.where(~ok)[0]
pairs = Counter((yte[i], pred[i]) for i in wrong).most_common(15)
def rel(a, b):
    A, B = CLASS_META[a], CLASS_META[b]
    if A['make'] == B['make'] and A['model'] == B['model']: return 'year'
    return 'same make' if A['make'] == B['make'] else 'different make'
pairs_df = pd.DataFrame([dict(true=CLASS_NAMES[a], predicted=CLASS_NAMES[b], count=c, relation=rel(a, b)) for (a, b), c in pairs])
RESULTS['similar_class_analysis']['top_confused_pairs'] = pairs_df.to_dict('records')
per_class = pd.DataFrame({'cls': yte, 'ok': ok}).groupby('cls').ok.mean().sort_values()
worst = pd.DataFrame({'class': [CLASS_NAMES[c] for c in per_class.index[:10]], 'accuracy': per_class.values[:10]})
RESULTS['similar_class_analysis']['hardest_classes'] = worst.to_dict('records'); save_results()
display(pairs_df); display(worst)

In [ ]:
# @title A.2 Visual example of the most-confused pair (true class vs predicted class)
(a, b), _ = pairs[0]
fig, axs = plt.subplots(2, 4, figsize=(18, 6))
for row, c in zip(axs, [a, b]):
    for ax_, (_, r) in zip(row, train_df[train_df.label == c].head(4).iterrows()):
        ax_.imshow(cv2.cvtColor(crop_box(cv2.imread(r.path), r[BOX].values, 0.05), cv2.COLOR_BGR2RGB)); ax_.axis('off')
    row[0].set_title(CLASS_NAMES[c], loc='left', fontsize=11)
plt.suptitle('Most-confused pair — training examples of both classes'); savefig('confused_pair'); plt.show()

## B. Requirements 2 & 3 — Illumination, viewing angle, occlusion, resolution
A random subset of unseen test images is perturbed **before detection** and the whole system is re-run (YOLO → crop → classifier → fusion). Reported per condition: localisation (IoU), deep-only top-1, hybrid top-1, and hybrid **with** the classical illumination normalisation.

In [ ]:
# @title B.1 Robustness suite (≈20–25 min)
def rotate_box(box, M):
    x1, y1, x2, y2 = box
    pts = np.array([[x1, y1, 1], [x2, y1, 1], [x1, y2, 1], [x2, y2, 1]]) @ M.T
    return [pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()]

def perturb(bgr, box, cond, rng):
    img = bgr.astype(np.float32); H, W = bgr.shape[:2]
    if cond == 'clean': return bgr, box
    if cond == 'dark': img = img * 0.3
    elif cond == 'very_dark': img = 255 * (img / 255) ** 3.0
    elif cond == 'overexposed': img = img * 1.6 + 60
    elif cond == 'low_contrast': img = (img - 128) * 0.35 + 128
    elif cond.startswith('occlusion'):
        frac = int(cond.split('_')[1]) / 100; x1, y1, x2, y2 = box
        bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
        a = rng.uniform(0.5, 2.0); ow = min(bw, math.sqrt(frac * bw * bh * a)); oh = min(bh, frac * bw * bh / ow)
        ox = x1 + rng.uniform(0, max(0.0, bw - ow)); oy = y1 + rng.uniform(0, max(0.0, bh - oh))
        img[int(oy):int(oy + oh), int(ox):int(ox + ow)] = rng.integers(0, 255, 3)
    elif cond.startswith('lowres'):
        f = float(cond.split('_')[1])
        small = cv2.resize(bgr, (max(8, int(W * f)), max(8, int(H * f))), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR), box
    elif cond == 'motion_blur':
        k = np.zeros((15, 15), np.float32); k[7, :] = 1 / 15; img = cv2.filter2D(img, -1, k)
    elif cond == 'gauss_noise': img = img + rng.normal(0, 25, img.shape)
    elif cond == 'rotate_15':
        M = cv2.getRotationMatrix2D((W / 2, H / 2), 15, 1.0)
        return cv2.warpAffine(bgr, M, (W, H), borderMode=cv2.BORDER_REFLECT), rotate_box(box, M)
    return np.clip(img, 0, 255).astype(np.uint8), box

CONDITIONS = ['clean', 'dark', 'very_dark', 'overexposed', 'low_contrast', 'occlusion_25', 'occlusion_40',
              'lowres_0.25', 'lowres_0.1', 'motion_blur', 'gauss_noise', 'rotate_15']
GROUP = {'dark': 'illumination', 'very_dark': 'illumination', 'overexposed': 'illumination', 'low_contrast': 'illumination',
         'occlusion_25': 'occlusion', 'occlusion_40': 'occlusion', 'lowres_0.25': 'resolution', 'lowres_0.1': 'resolution',
         'motion_blur': 'image quality', 'gauss_noise': 'image quality', 'rotate_15': 'viewing angle', 'clean': 'baseline'}
rob_idx = np.random.default_rng(SEED).choice(len(test_df), min(CFG['N_ROBUST'], len(test_df)), replace=False)

@torch.no_grad()
def classify_crops(crops, enhance):
    xs = []
    for c in crops:
        if enhance: c, _ = adaptive_enhance(c, CFG['DARK_THR'], CFG['BRIGHT_THR'])
        xs.append(eval_tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))))
    x = torch.stack(xs).to(DEVICE).to(memory_format=torch.channels_last)
    with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
        return ((model(x).float() + model(torch.flip(x, [3])).float()) / 2).cpu().numpy()

ROB_CACHE = f'{CACHE}/robustness.json'
rob_rows = json.load(open(ROB_CACHE)) if os.path.exists(ROB_CACHE) else []
done = {r['condition'] for r in rob_rows}; rob_examples = {}
with Pool(os.cpu_count()) as pool:
    for cond in CONDITIONS:
        rng = np.random.default_rng(SEED); zs, zs_e, lps, ious, ys = [], [], [], [], []
        for k in tqdm(range(0, len(rob_idx), 32), desc=cond, leave=False):
            imgs, boxes, labs = [], [], []
            for i in rob_idx[k:k + 32]:
                bgr = cv2.imread(test_df.path[i]); s = min(1.0, 800 / max(bgr.shape[:2]))
                bgr = cv2.resize(bgr, None, fx=s, fy=s) if s < 1 else bgr
                im, bx = perturb(bgr, (GT_TEST[i] * s).tolist(), cond, rng)
                imgs.append(im); boxes.append(bx); labs.append(yte[i])
            if cond not in rob_examples: rob_examples[cond] = cv2.cvtColor(imgs[0], cv2.COLOR_BGR2RGB)
            if cond in done: break           # only need the example image
            res = det_ft.predict(imgs, conf=0.01, iou=0.6, imgsz=CFG['YOLO_IMGSZ'], half=DEVICE == 'cuda', verbose=False)
            crops = []
            for im, r, g in zip(imgs, res, boxes):
                b, sc = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
                pb = b[np.argmax(sc)].tolist() if len(b) else classical_roi(im)[0]
                ious.append(iou_one(pb, g)[0] if pb is not None else 0.0)
                crops.append(crop_box(im, pb, CFG['CROP_PAD']))
            zs.append(classify_crops(crops, False)); zs_e.append(classify_crops(crops, True))
            lps.append(classical_logp(np.stack(pool.map(extract_classical_features, crops)))); ys += labs
        if cond in done: continue
        zc, zce, lpc, ys = np.concatenate(zs), np.concatenate(zs_e), np.concatenate(lps), np.array(ys)
        md, mh, mhe = cls_metrics(zc, ys), cls_metrics(hybrid_probs(zc, lpc), ys), cls_metrics(hybrid_probs(zce, lpc), ys)
        rob_rows.append(dict(condition=cond, group=GROUP[cond], mean_IoU=float(np.mean(ious)), loc_acc=float(np.mean(np.array(ious) >= 0.5)),
                             deep_top1=md['top1'], hybrid_top1=mh['top1'], hybrid_enh_top1=mhe['top1'], hybrid_enh_top5=mhe['top5'],
                             hybrid_enh_make_acc=mhe['make_acc']))
        json.dump(rob_rows, open(ROB_CACHE, 'w'))
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rob_rows[-1].items()})
rob_df = pd.DataFrame(rob_rows).set_index('condition').loc[CONDITIONS]
rob_df['drop_vs_clean'] = rob_df.hybrid_enh_top1 - rob_df.loc['clean', 'hybrid_enh_top1']
RESULTS['robustness'] = rob_df.to_dict('index'); save_results()
rob_df.round(4)

In [ ]:
# @title B.2 Robustness plots
ax = rob_df[['deep_top1', 'hybrid_top1', 'hybrid_enh_top1', 'loc_acc']].plot.bar(figsize=(16, 4.5)); ax.set_ylim(0, 1)
ax.legend(['deep only', 'hybrid', 'hybrid + illumination norm.', 'localisation (IoU≥0.5)'], ncol=4, loc='lower left')
plt.title('End-to-end robustness on unseen test images'); plt.xticks(rotation=30); savefig('robustness_bars'); plt.show()
fig, axs = plt.subplots(2, 6, figsize=(22, 6))
for ax_, c in zip(axs.ravel(), CONDITIONS):
    ax_.imshow(rob_examples[c]); ax_.set_title(f"{c}\nacc {rob_df.loc[c,'hybrid_enh_top1']:.1%}", fontsize=10); ax_.axis('off')
savefig('robustness_examples'); plt.show()
gain = (rob_df.hybrid_enh_top1 - rob_df.deep_top1).round(4)
print('Gain of hybrid (classical illumination norm + fusion) over deep-only, per condition:\n', gain.to_string())

In [ ]:
# @title B.3 Viewing angle / resolution / brightness slices (clean test set)
ar = (GT_TEST[:, 2] - GT_TEST[:, 0]) / (GT_TEST[:, 3] - GT_TEST[:, 1])
side = np.sqrt((GT_TEST[:, 2] - GT_TEST[:, 0]) * (GT_TEST[:, 3] - GT_TEST[:, 1]))
slices = {
    'viewing angle (bbox aspect ratio)': pd.cut(ar, [0, 1.3, 1.8, 10], labels=['front/rear (<1.3)', 'three-quarter (1.3-1.8)', 'side (>1.8)']),
    'vehicle resolution (sqrt of box area, px)': pd.qcut(side, 4, labels=['smallest 25%', 'Q2', 'Q3', 'largest 25%']),
    'brightness (HSV-V of vehicle)': pd.qcut(TEST_BRIGHT, 4, labels=['darkest 25%', 'Q2', 'Q3', 'brightest 25%']),
}
fig, axs = plt.subplots(1, 3, figsize=(18, 3.8)); RESULTS['slices'] = {}
for ax_, (name, bins) in zip(axs, slices.items()):
    s = pd.DataFrame({'bin': bins, 'acc': ok, 'iou': IOU_FT}).groupby('bin', observed=True).agg(top1=('acc', 'mean'), IoU=('iou', 'mean'), n=('acc', 'size'))
    RESULTS['slices'][name] = s.to_dict('index'); s[['top1', 'IoU']].plot.bar(ax=ax_, ylim=(0.7, 1), title=name, rot=15)
    print(name, '\n', s.round(3), '\n')
save_results(); savefig('slices'); plt.show()

## C. Confidence analysis + fixed low-confidence thresholds
The training notebook's 95 % target gave `tau = 0.14` and `tau_make = 0`, so the "low confidence" path never triggered. Here the targets are **99 %** for confident predictions and **98 %** for make-only predictions, with a floor of 0.30.

In [ ]:
# @title C.1 Calibration (reliability diagrams, ECE)
def ece(probs, y, n_bins=15):
    c, p = probs.max(1), probs.argmax(1); bins = np.linspace(0, 1, n_bins + 1); e = 0; rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (c > lo) & (c <= hi)
        if m.any():
            acc, cm = (p[m] == y[m]).mean(), c[m].mean(); e += m.mean() * abs(acc - cm); rows.append((cm, acc, m.sum()))
    return float(e), np.array(rows)
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5)); RESULTS['ece'] = {}
for ax_, (nm, p) in zip(axs, [('deep raw', softmax(z_te, 1)), ('deep + temperature', softmax(z_te / T_DEEP, 1)), ('hybrid (final)', P_TEST)]):
    e, rows = ece(p, yte); RESULTS['ece'][nm] = e
    ax_.plot([0, 1], [0, 1], 'k--'); ax_.bar(rows[:, 0], rows[:, 1], width=0.05, alpha=.7)
    ax_.set_title(f'{nm}  ECE={e:.3f}'); ax_.set_xlabel('confidence'); ax_.set_ylabel('accuracy')
save_results(); savefig('reliability'); plt.show()

In [ ]:
# @title C.2 Thresholds (chosen on validation, applied to test) + low-confidence handling
def make_probs(P): return np.stack([P[:, MAKE_ID == m].sum(1) for m in range(len(MAKES))], 1)
def pick_threshold(c, correct, target, floor):
    for t in np.linspace(floor, 0.99, 70):
        m = c >= t
        if m.sum() >= 20 and correct[m].mean() >= target: return float(t)
    return 0.9
TAU = pick_threshold(P_VAL.max(1), P_VAL.argmax(1) == yval, CFG['TARGET_PRECISION'], CFG['MIN_TAU'])
MPV = make_probs(P_VAL)
TAU_MAKE = pick_threshold(MPV.max(1), MPV.argmax(1) == MAKE_ID[yval], CFG['TAU_MAKE_TARGET'], CFG['MIN_TAU'])
print(f'tau = {TAU:.2f}   tau_make = {TAU_MAKE:.2f}')

ts = np.linspace(0, 0.99, 100)
cov = [(conf >= t).mean() for t in ts]; accs = [ok[conf >= t].mean() if (conf >= t).any() else np.nan for t in ts]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(ts, accs, label='accuracy of accepted'); ax[0].plot(ts, cov, label='coverage'); ax[0].axvline(TAU, c='r', ls='--', label=f'tau={TAU:.2f}')
ax[0].legend(); ax[0].set_xlabel('confidence threshold'); ax[0].set_title('Threshold analysis (unseen test)')
ax[1].hist(conf[ok], 40, alpha=.6, label='correct'); ax[1].hist(conf[~ok], 40, alpha=.6, label='wrong'); ax[1].set_yscale('log')
ax[1].legend(); ax[1].set_title('Confidence of correct vs wrong predictions'); savefig('threshold_analysis'); plt.show()

MPT = make_probs(P_TEST)
status = np.where(conf >= TAU, 'confident', np.where(MPT.max(1) >= TAU_MAKE, 'uncertain_model', 'low_confidence'))
dec = pd.DataFrame({'status': status, 'model_correct': ok, 'make_correct': MPT.argmax(1) == MAKE_ID[yte]}) \
    .groupby('status').agg(share=('model_correct', lambda s: len(s) / n), model_acc=('model_correct', 'mean'), make_acc=('make_correct', 'mean'))
thr_table = pd.DataFrame([dict(threshold=round(t, 1), coverage=float((conf >= t).mean()),
                               accuracy=float(ok[conf >= t].mean()) if (conf >= t).any() else None) for t in np.arange(0, 1, 0.1)])
RESULTS['confidence'] = dict(tau=TAU, tau_make=TAU_MAKE, target_precision=CFG['TARGET_PRECISION'],
                             decision_breakdown=dec.to_dict('index'), threshold_table=thr_table.to_dict('records'))
save_results(); display(dec.round(4)); display(thr_table.round(4))

## D. Requirement 6 — Model optimisation (reduced precision, quantisation, pruning)
Applied to the trained classifier **without retraining**. Size, batch-1 latency and top-1 on a fixed subset of unseen test vehicles.

In [ ]:
# @title D.1 FP32 vs FP16 vs ONNX vs INT8 vs pruning
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import torch.nn.utils.prune as prune

opt_idx = np.random.default_rng(1).choice(len(test_df), min(CFG['N_OPT_EVAL'], len(test_df)), replace=False)
opt_loader = DataLoader(CarDS(test_df.path.values[opt_idx], yte[opt_idx], boxes=GT_TEST[opt_idx].tolist()), 50, num_workers=NW)
XO = [(x.half(), y.numpy()) for x, y in tqdm(opt_loader, desc='load subset', leave=False)]; yo = np.concatenate([y for _, y in XO])

@torch.no_grad()
def acc_torch(m, device, half=False):
    m = m.to(device).eval(); out = []
    for x, _ in XO:
        x = x.to(device); out.append(m(x.half() if half else x.float()).float().cpu().numpy())
    return cls_metrics(np.concatenate(out), yo)['top1']
def acc_onnx(sess): return cls_metrics(np.concatenate([sess.run(None, {'input': x.float().numpy()})[0] for x, _ in XO]), yo)['top1']

rows = []
fp32 = copy.deepcopy(model).float().eval().to(memory_format=torch.contiguous_format)
torch.save(fp32.state_dict(), f'{WORK}/fp32.pth'); x1g = torch.randn(1, 3, S, S, device=DEVICE)
with torch.no_grad():
    rows.append(dict(variant='PyTorch FP32 (GPU)', technique='baseline', size_mb=os.path.getsize(f'{WORK}/fp32.pth') / 1e6,
                     ms=bench(lambda: fp32.to(DEVICE)(x1g)), top1=acc_torch(fp32, DEVICE)))
    fp16 = copy.deepcopy(fp32).half().to(DEVICE)
    torch.save(dict(state_dict=fp16.state_dict(), model_name=MODEL_NAME, num_classes=NUM_CLASSES, img_size=S, mean=MEAN, std=STD), f'{ART}/classifier_fp16.pth')
    rows.append(dict(variant='PyTorch FP16 (GPU)', technique='reduced precision', size_mb=os.path.getsize(f'{ART}/classifier_fp16.pth') / 1e6,
                     ms=bench(lambda: fp16(x1g.half())), top1=acc_torch(fp16, DEVICE, half=True)))
    cpu_m = copy.deepcopy(fp32).cpu(); x1c = torch.randn(1, 3, S, S)
    rows.append(dict(variant='PyTorch FP32 (CPU)', technique='baseline', size_mb=rows[0]['size_mb'],
                     ms=bench(lambda: cpu_m(x1c), n=10, warm=2), top1=rows[0]['top1']))

ONNX32, ONNX8 = f'{ART}/classifier_fp32.onnx', f'{ART}/classifier_int8.onnx'
kw = dict(input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
try: torch.onnx.export(cpu_m, x1c, ONNX32, dynamo=False, **kw)
except TypeError: torch.onnx.export(cpu_m, x1c, ONNX32, **kw)
quantize_dynamic(ONNX32, ONNX8, weight_type=QuantType.QInt8, op_types_to_quantize=['MatMul', 'Gemm'])
for nm, tech, pth in [('ONNX Runtime FP32 (CPU)', 'graph export', ONNX32), ('ONNX Runtime INT8 (CPU)', 'dynamic INT8 quantisation', ONNX8)]:
    sess = ort.InferenceSession(pth, providers=['CPUExecutionProvider']); xn = x1c.numpy()
    rows.append(dict(variant=nm, technique=tech, size_mb=os.path.getsize(pth) / 1e6,
                     ms=bench(lambda: sess.run(None, {'input': xn}), n=20, warm=3), top1=acc_onnx(sess)))
for amount in [0.3, 0.5]:
    pm = copy.deepcopy(fp32)
    params = [(m_, 'weight') for m_ in pm.modules() if isinstance(m_, (nn.Conv2d, nn.Linear))]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    for m_, _ in params: prune.remove(m_, 'weight')
    nz = sum((m_.weight != 0).sum().item() for m_, _ in params); tot = sum(m_.weight.numel() for m_, _ in params)
    rows.append(dict(variant=f'L1 pruning {int(amount*100)}% (no fine-tune)', technique='pruning',
                     size_mb=rows[0]['size_mb'] * nz / tot, ms=np.nan, top1=acc_torch(pm, DEVICE), sparsity=1 - nz / tot))
def yolo_ms(half):
    im = cv2.imread(test_df.path[0])
    return bench(lambda: det_ft.predict(im, imgsz=CFG['YOLO_IMGSZ'], half=half, verbose=False), n=20, warm=3)
rows.append(dict(variant='YOLOv8s detector FP32', technique='baseline', size_mb=os.path.getsize(YOLO_BEST) / 1e6, ms=yolo_ms(False), top1=np.nan))
rows.append(dict(variant='YOLOv8s detector FP16', technique='reduced precision', size_mb=os.path.getsize(YOLO_BEST) / 1e6 / 2, ms=yolo_ms(True), top1=np.nan))
opt_df = pd.DataFrame(rows).set_index('variant')
RESULTS['optimization'] = opt_df.to_dict('index'); save_results()
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8)); cl = opt_df[opt_df.top1.notna()]
cl.size_mb.plot.barh(ax=ax[0], title='Classifier size (MB)'); cl.top1.plot.barh(ax=ax[1], xlim=(max(0, cl.top1.min() - 0.05), 1), title='Top-1 on subset')
savefig('optimization'); plt.show()
opt_df.round(4)

## E. Requirement 8 — Failure cases and their causes
Each wrong prediction gets one cause (checked in this order): localisation error → low resolution → low light → truncation/occlusion → year confusion → same-make confusion → cross-make look-alike. The gallery shows the most confident mistakes per cause with Grad-CAM.

In [ ]:
# @title E.1 Cause tagging
BODY = ['Sedan', 'SUV', 'Coupe', 'Convertible', 'Hatchback', 'Wagon', 'Minivan', 'Van', 'Cab']
def body_type(c): return next((b for b in BODY if b in CLASS_META[c]['model']), 'other')
min_side = np.minimum(GT_TEST[:, 2] - GT_TEST[:, 0], GT_TEST[:, 3] - GT_TEST[:, 1])
truncated = (GT_TEST[:, 0] <= 2) | (GT_TEST[:, 1] <= 2) | (GT_TEST[:, 2] >= test_df.W.values - 3) | (GT_TEST[:, 3] >= test_df.H.values - 3)
def failure_cause(i, p):
    G, P = CLASS_META[yte[i]], CLASS_META[p]
    if IOU_FT[i] < 0.5: return 'localisation', f'Detector box IoU={IOU_FT[i]:.2f} with ground truth: the crop missed part of the car or included another object, so the classifier saw an incomplete vehicle.'
    if min_side[i] < 100: return 'low_resolution', f'Vehicle is only ~{int(min_side[i])} px on its short side; badges, grille and lamp details that separate similar classes are lost.'
    if TEST_BRIGHT[i] < 70: return 'low_light', f'Vehicle brightness {TEST_BRIGHT[i]:.0f}/255: under-exposure hides texture and colour cues even after CLAHE.'
    if truncated[i]: return 'truncation/occlusion', 'Vehicle is cut off by the image border, so distinguishing front/rear parts are partially missing.'
    if G['make'] == P['make'] and G['model'] == P['model']: return 'year_confusion', f'Same {G["make"]} {G["model"]}, different year ({G["year"]} vs {P["year"]}): generations differ only in small facelift details.'
    if G['make'] == P['make']: return 'same_make', f'Both are {G["make"]} vehicles with shared brand design (grille, headlights); the difference is in body proportions/details.'
    return 'cross_make_lookalike', f'Different makes but similar silhouette ({body_type(yte[i])} vs {body_type(p)}) and colour; the viewpoint hides brand-specific cues.'
causes = [failure_cause(i, pred[i]) for i in wrong]
cnt = Counter(c for c, _ in causes)
RESULTS['failure_causes'] = dict(cnt); save_results()
plt.figure(figsize=(8, 3)); plt.barh(list(cnt.keys()), list(cnt.values()))
plt.title(f'Failure causes ({len(wrong)} errors of {n} unseen test images)'); savefig('failure_causes'); plt.show()
print(cnt)

In [ ]:
# @title E.2 Failure gallery with Grad-CAM
class GradCAM:
    def __init__(self, m):
        self.m = m; layer = m.stages[-1] if hasattr(m, 'stages') else list(m.children())[-3]
        layer.register_forward_hook(self._fwd)
    def _fwd(self, mod, inp, out):
        self.act = out
        if out.requires_grad: out.register_hook(lambda g: setattr(self, 'grad', g))
    def __call__(self, x):
        self.m.zero_grad(); out = self.m(x.clone().requires_grad_(True)); out[0, int(out.argmax(1))].backward()
        w = self.grad.mean((2, 3), keepdim=True); cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, x.shape[-2:], mode='bilinear')[0, 0].detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
gcam = GradCAM(copy.deepcopy(model).float().eval().to(memory_format=torch.contiguous_format))
def cam_overlay(crop):
    cam = gcam(eval_tf(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))[None].to(DEVICE))
    rgb = cv2.cvtColor(cv2.resize(crop, (S, S)), cv2.COLOR_BGR2RGB)
    heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return (0.55 * rgb + 0.45 * heat).astype(np.uint8)

by_cause = {}
for j in np.argsort(-conf[wrong]):
    i = wrong[j]; c, why = failure_cause(i, pred[i])
    if len(by_cause.setdefault(c, [])) < 2: by_cause[c].append((i, why))
picked = [v for vs in by_cause.values() for v in vs][:8]
fig, axs = plt.subplots(len(picked), 3, figsize=(16, 4.3 * len(picked))); axs = np.atleast_2d(axs)
RESULTS['failure_examples'] = []
for row, (i, why) in zip(axs, picked):
    im = cv2.imread(test_df.path[i]); vis = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).copy()
    g, b = GT_TEST[i], TEST_BOXES_YOLO[i]
    cv2.rectangle(vis, (int(g[0]), int(g[1])), (int(g[2]), int(g[3])), (0, 255, 0), 4)
    cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 3)
    ex = train_df[train_df.label == pred[i]].iloc[0]
    row[0].imshow(vis); row[0].set_title(f'TRUE: {CLASS_NAMES[yte[i]]}  (green=GT, red=YOLO)', fontsize=9)
    row[1].imshow(cam_overlay(crop_box(im, b, CFG['CROP_PAD']))); row[1].set_title(f'PRED: {CLASS_NAMES[pred[i]]} ({conf[i]:.0%}) — Grad-CAM', fontsize=9)
    row[2].imshow(cv2.cvtColor(crop_box(cv2.imread(ex.path), ex[BOX].values, 0.05), cv2.COLOR_BGR2RGB)); row[2].set_title('Training example of the predicted class', fontsize=9)
    row[0].set_xlabel(why, fontsize=8)
    for a_ in row: a_.set_xticks([]); a_.set_yticks([])
    RESULTS['failure_examples'].append(dict(file=test_df.fname[i], true=CLASS_NAMES[yte[i]], pred=CLASS_NAMES[pred[i]],
                                            conf=float(conf[i]), cause=failure_cause(i, pred[i])[0], explanation=why))
plt.tight_layout(); savefig('failure_gallery'); plt.show(); save_results()
for e in RESULTS['failure_examples']: print(f"- [{e['cause']}] {e['true']} -> {e['pred']} ({e['conf']:.0%}): {e['explanation']}")

## F. Files for the frontend + end-to-end test
Writes `config.json` (with the fixed thresholds), `class_meta.json`, `inference.py`, `requirements.txt` into `artifacts/`, then runs the exact pipeline the frontend will use.

In [ ]:
# @title F.1 config.json + class_meta.json
CONFIG = dict(model_name=MODEL_NAME, num_classes=NUM_CLASSES, img_size=S, mean=MEAN, std=STD, crop_pad=CFG['CROP_PAD'],
              T=T_DEEP, alpha=ALPHA, T2=T2, tau=TAU, tau_make=TAU_MAKE, det_conf=CFG['DET_CONF'], yolo_imgsz=CFG['YOLO_IMGSZ'],
              enhance=True, dark_thr=CFG['DARK_THR'], bright_thr=CFG['BRIGHT_THR'], tta=True)
json.dump(CONFIG, open(f'{ART}/config.json', 'w'), indent=2)
json.dump(CLASS_META, open(f'{ART}/class_meta.json', 'w'), indent=1)
open(f'{ART}/requirements.txt', 'w').write('torch\ntorchvision\ntimm\nultralytics\nopencv-python-headless\nscikit-image\nscikit-learn\njoblib\nnumpy\nPillow\nonnxruntime\n')
CONFIG

In [ ]:
%%writefile /content/work/inference.py
"""P5 Fine-Grained Vehicle Recognition — inference module for the frontend.

    from inference import VehicleRecognizer
    rec = VehicleRecognizer('artifacts')          # backend='onnx' -> CPU INT8 classifier
    out = rec.predict('car.jpg')

Needs in the artifacts folder: config.json, class_meta.json, yolo_car_best.pt, classifier_best.pth
(or classifier_int8.onnx), classical_lr.joblib, and classical_cv.py next to this file.
"""
import os, json, time
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
from classical_cv import to_bgr, adaptive_enhance, crop_box, classical_roi, extract_classical_features, processing_stages


def _log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


STATUS_COLORS = {'confident': (0, 200, 0), 'uncertain_model': (255, 165, 0), 'low_confidence': (230, 0, 0)}


class VehicleRecognizer:
    def __init__(self, art_dir='artifacts', device=None, backend='torch', onnx_file='classifier_int8.onnx', use_fusion=True):
        from ultralytics import YOLO
        self.art = art_dir
        self.cfg = json.load(open(os.path.join(art_dir, 'config.json')))
        self.classes = json.load(open(os.path.join(art_dir, 'class_meta.json')))
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.backend = backend
        self.det = YOLO(os.path.join(art_dir, 'yolo_car_best.pt'))
        S = self.cfg['img_size']
        self.tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(self.cfg['mean'], self.cfg['std'])])
        self.clf = None
        if backend == 'onnx':
            import onnxruntime as ort
            self.sess = ort.InferenceSession(os.path.join(art_dir, onnx_file), providers=['CPUExecutionProvider'])
        else:
            import timm
            ck = torch.load(os.path.join(art_dir, 'classifier_best.pth'), map_location='cpu', weights_only=False)
            self.clf = timm.create_model(ck['model_name'], pretrained=False, num_classes=ck['num_classes'])
            self.clf.load_state_dict(ck['state_dict'])
            self.clf.eval().to(self.device)
        self.classical = None
        p = os.path.join(art_dir, 'classical_lr.joblib')
        if use_fusion and self.cfg.get('alpha', 0) > 0 and os.path.exists(p):
            import joblib
            self.classical = joblib.load(p)
        self.make_names = sorted({c['make'] for c in self.classes})
        self.make_idx = np.array([self.make_names.index(c['make']) for c in self.classes])
        self._cam = None

    # ---------- detection with low-confidence fallbacks ----------
    def detect(self, bgr, max_vehicles=5):
        r = self.det.predict(bgr, conf=0.05, iou=0.6, imgsz=self.cfg['yolo_imgsz'], verbose=False)[0]
        b, s = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
        order = np.argsort(-s)
        keep = [i for i in order if s[i] >= self.cfg['det_conf']][:max_vehicles]
        if keep:
            return [(b[i].tolist(), float(s[i]), 'yolo') for i in keep]
        if len(b):
            i = order[0]
            return [(b[i].tolist(), float(s[i]), 'yolo_low_confidence')]
        box, score, _ = classical_roi(bgr)
        if box is not None:
            return [(list(map(float, box)), score, 'classical_roi_fallback')]
        H, W = bgr.shape[:2]
        return [([0.0, 0.0, float(W), float(H)], 0.0, 'full_image_fallback')]

    # ---------- classification ----------
    def _logits(self, crops):
        x = torch.stack([self.tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops])
        if self.backend == 'onnx':
            xn = x.numpy()
            z = self.sess.run(None, {'input': xn})[0]
            if self.cfg.get('tta', True):
                z = (z + self.sess.run(None, {'input': np.ascontiguousarray(xn[:, :, :, ::-1])})[0]) / 2
            return z
        x = x.to(self.device)
        with torch.no_grad():
            z = self.clf(x).float()
            if self.cfg.get('tta', True):
                z = (z + self.clf(torch.flip(x, [3])).float()) / 2
        return z.cpu().numpy()

    def classify(self, crops):
        enh = [adaptive_enhance(c, self.cfg['dark_thr'], self.cfg['bright_thr'])[0] if self.cfg.get('enhance') else c for c in crops]
        s = _log_softmax(self._logits(enh) / self.cfg['T'])
        if self.classical is not None:
            f = np.stack([extract_classical_features(c) for c in crops])
            b = self.classical
            s = s + self.cfg['alpha'] * b['lr'].predict_log_proba(b['pca'].transform(b['scaler'].transform(f)))
        return np.exp(_log_softmax(s / self.cfg['T2']))

    def decide(self, p):
        top = np.argsort(-p)[:5]
        c = self.classes[top[0]]
        mp = np.bincount(self.make_idx, weights=p, minlength=len(self.make_names))
        mk = int(mp.argmax())
        conf = float(p[top[0]])
        cands = ', '.join(self.classes[i]['name'] for i in top[:3])
        if conf >= self.cfg['tau']:
            status, msg = 'confident', f"{c['make']} {c['model']} ({c['year']}) — {conf:.1%}"
        elif mp[mk] >= self.cfg['tau_make']:
            status, msg = 'uncertain_model', f"Likely a {self.make_names[mk]} ({mp[mk]:.0%}); exact model/year uncertain. Candidates: {cands}"
        else:
            status, msg = 'low_confidence', f"Low confidence — try a clearer / closer image. Top guesses: {cands}"
        return dict(label=c['name'], make=c['make'], model=c['model'], year=c['year'], class_id=int(top[0]),
                    confidence=conf, status=status, message=msg,
                    make_guess=self.make_names[mk], make_confidence=float(mp[mk]),
                    top5=[dict(label=self.classes[i]['name'], confidence=float(p[i])) for i in top])

    # ---------- explainability ----------
    def gradcam(self, crop_bgr):
        if self.clf is None:
            return None
        if self._cam is None:
            layer = self.clf.stages[-1] if hasattr(self.clf, 'stages') else list(self.clf.children())[-3]
            def hook(mod, inp, out):
                self._act = out
                if out.requires_grad:
                    out.register_hook(lambda g: setattr(self, '_grad', g))
            layer.register_forward_hook(hook)
            self._cam = True
        x = self.tf(Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)))[None].to(self.device)
        self.clf.zero_grad()
        with torch.enable_grad():
            out = self.clf(x.requires_grad_(True))
            out[0, int(out.argmax(1))].backward()
        w = self._grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * self._act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=crop_bgr.shape[:2], mode='bilinear')[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        return cv2.cvtColor((0.55 * crop_bgr + 0.45 * heat).astype(np.uint8), cv2.COLOR_BGR2RGB)

    # ---------- drawing ----------
    @staticmethod
    def draw(bgr, vehicles):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).copy()
        th = max(2, int(max(rgb.shape[:2]) / 300))
        for v in vehicles:
            x1, y1, x2, y2 = [int(t) for t in v['box']]
            col = STATUS_COLORS[v['status']]
            cv2.rectangle(rgb, (x1, y1), (x2, y2), col, th)
            txt = f"{v['label']} {v['confidence']:.0%}" if v['status'] == 'confident' else f"{v['make_guess']}? {v['make_confidence']:.0%}"
            fs = max(0.5, th / 3)
            (tw, tht), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs, max(1, th // 2))
            yt = max(tht + 6, y1)
            cv2.rectangle(rgb, (x1, yt - tht - 6), (x1 + tw + 6, yt), col, -1)
            cv2.putText(rgb, txt, (x1 + 3, yt - 4), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), max(1, th // 2), cv2.LINE_AA)
        return rgb

    # ---------- full pipeline ----------
    def predict(self, image, max_vehicles=5, return_stages=True, gradcam=False):
        t0 = time.time()
        bgr = to_bgr(image)
        dets = self.detect(bgr, max_vehicles)
        t1 = time.time()
        crops = [crop_box(bgr, d[0], self.cfg['crop_pad']) for d in dets]
        probs = self.classify(crops)
        t2 = time.time()
        vehicles = []
        for (box, dconf, src), p in zip(dets, probs):
            v = self.decide(p)
            v.update(box=[float(t) for t in box], det_conf=dconf, det_source=src)
            vehicles.append(v)
        out = dict(vehicles=vehicles, annotated_image=self.draw(bgr, vehicles),
                   timings_ms=dict(detection=(t1 - t0) * 1000, classification=(t2 - t1) * 1000))
        if return_stages:
            out['stages'], out['classical_info'] = processing_stages(bgr, dets[0][0])
        if gradcam:
            out['gradcam'] = self.gradcam(crops[0])
        out['timings_ms']['total'] = (time.time() - t0) * 1000
        return out

In [ ]:
# @title F.2 Test the exported pipeline (what the frontend runs)
shutil.copy(f'{WORK}/inference.py', ART)
import inference; importlib.reload(inference)
from inference import VehicleRecognizer
rec = VehicleRecognizer(ART)

def show_result(out, title=''):
    has_cam = out.get('gradcam') is not None
    fig, axs = plt.subplots(1, 3 if has_cam else 2, figsize=(18, 5))
    axs[0].imshow(out['annotated_image']); axs[0].set_title(title); axs[0].axis('off')
    v = out['vehicles'][0]
    axs[1].barh([t['label'] for t in v['top5']][::-1], [t['confidence'] for t in v['top5']][::-1]); axs[1].set_xlim(0, 1)
    axs[1].set_title(f"{v['status']}: {v['message']}", fontsize=9)
    if has_cam: axs[2].imshow(out['gradcam']); axs[2].set_title('Grad-CAM'); axs[2].axis('off')
    plt.tight_layout(); plt.show()

lat = []
for i in [0, 10, 20, 30]:
    out = rec.predict(test_df.path[i], gradcam=True); lat.append(out['timings_ms']['total'])
    show_result(out, f'GT: {CLASS_NAMES[yte[i]]}')
    print({k: v for k, v in out['vehicles'][0].items() if k != 'top5'}, {k: round(v, 1) for k, v in out['timings_ms'].items()})
RESULTS['end_to_end_ms_per_image'] = float(np.mean(lat[1:]))
st = out['stages']; fig, axs = plt.subplots(3, 4, figsize=(18, 10))
for a_, (k, v) in zip(axs.ravel(), st.items()): a_.imshow(v); a_.set_title(k, fontsize=9)
for a_ in axs.ravel(): a_.axis('off')
plt.suptitle('Intermediate image-processing results (shown in the app)'); savefig('pipeline_stages'); plt.show()
rec_onnx = VehicleRecognizer(ART, backend='onnx')
t0 = time.time(); o = rec_onnx.predict(test_df.path[0], return_stages=False)
RESULTS['end_to_end_ms_onnx_int8'] = (time.time() - t0) * 1000
print('ONNX INT8 backend:', o['vehicles'][0]['message']); save_results()

## G. Requirement checklist + export

In [ ]:
# @title G.1 Summary of all 8 requirements
R = RESULTS; fs = R.get('final_system_test', {}); h = R.get('similar_class_analysis', {}).get('error_hierarchy', {})
rb = R.get('robustness', {}); op = R.get('optimization', {})
def rbv(c): return f"{rb[c]['hybrid_enh_top1']:.1%}" if c in rb else 'n/a'
int8 = op.get('ONNX Runtime INT8 (CPU)', {}); f32 = op.get('ONNX Runtime FP32 (CPU)', {})
lines = [
 ('1. Similar makes/models/years', f"top-1 {fs.get('top1',0):.1%}, make {fs.get('make_acc',0):.1%}; errors: year-only {h.get('wrong year only (same make+model)',0):.2%}, same-make {h.get('same make, wrong model',0):.2%}, other make {h.get('wrong make',0):.2%}"),
 ('2. Illumination & viewing angle', f"dark {rbv('dark')}, very dark {rbv('very_dark')}, overexposed {rbv('overexposed')}, rotated {rbv('rotate_15')} (clean {rbv('clean')})"),
 ('3. Occlusion & resolution', f"occl. 25% {rbv('occlusion_25')}, 40% {rbv('occlusion_40')}, ×0.25 res {rbv('lowres_0.25')}, ×0.1 res {rbv('lowres_0.1')}"),
 ('4. Classical vs deep vs hybrid', ' | '.join(f"{k.split(' (')[0]}: {v['top1']:.1%}" for k, v in R.get('approach_comparison', {}).items())),
 ('5. Ablation', f"{len(R.get('ablation', {}))} configurations in metrics.json['ablation']"),
 ('6. Optimisation', f"INT8 ONNX {int8.get('size_mb',0):.0f} MB vs FP32 {f32.get('size_mb',0):.0f} MB, top-1 {int8.get('top1',0):.1%} vs {f32.get('top1',0):.1%}; FP16 + pruning also reported"),
 ('7. Unseen test images', f"all metrics on {len(test_df)} test images never used for training/tuning"),
 ('8. Failure cases', f"{len(R.get('failure_examples', []))} explained examples; causes: {dict(R.get('failure_causes', {}))}"),
]
summary = pd.DataFrame(lines, columns=['requirement', 'result']); RESULTS['requirements_summary'] = dict(lines); save_results()
with open(f'{ART}/results_summary.md', 'w') as fh:
    fh.write('| Requirement | Result |\n|---|---|\n' + '\n'.join(f'| {a} | {b} |' for a, b in lines) + '\n')
pd.set_option('display.max_colwidth', None); summary

In [ ]:
# @title G.2 Zip + download (artifacts + plots)
shutil.make_archive(f'{OUT}/P5_artifacts', 'zip', ART)
shutil.make_archive(f'{OUT}/P5_plots', 'zip', PLOTS)
for f in sorted(os.listdir(ART)): print(f'{f:28s} {os.path.getsize(f"{ART}/{f}")/1e6:8.2f} MB')
try:
    from google.colab import files
    files.download(f'{OUT}/P5_artifacts.zip'); files.download(f'{OUT}/P5_plots.zip')
except Exception as e:
    print('Download manually from Drive:', OUT, e)